# SuperPoint + LightGlue: Reference-crop matching

Given a **reference crop** (recorded UI element) and a **target image** (live screen), locate the matching region.

Demoed on `github_light.png` → `github_dark.png` to show robustness across themes.

## 0. Environment

Deps are pinned in `notebooks/pyproject.toml` + `notebooks/uv.lock`.

```sh
cd notebooks
uv sync
uv run jupyter lab   # or point your kernel at notebooks/.venv/bin/python
```

LightGlue ships SuperPoint inside it.

In [ ]:
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import torch
from lightglue import LightGlue, SuperPoint
from lightglue.utils import load_image, rbd
from PIL import Image

DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print(f"device: {DEVICE}")

## 1. Inputs

- `REFERENCE_IMAGE`: the full screenshot the crop is taken from (record-time frame).
- `REFERENCE_BBOX`: `(x, y, w, h)` of the recorded UI element inside that screenshot.
- `TARGET_IMAGE`: the screenshot to search in (replay-time frame).

Adjust `REFERENCE_BBOX` to point at any element in `github_light.png`.

In [ ]:
NOTEBOOK_DIR = Path.cwd()
REFERENCE_IMAGE = NOTEBOOK_DIR / "github_light.png"
TARGET_IMAGE = NOTEBOOK_DIR / "github_dark.png"

# (x, y, w, h) in pixels of the recorded element inside REFERENCE_IMAGE.
# Pick something distinctive — a button, an icon with text, etc.
REFERENCE_BBOX = (0, 0, 400, 120)  # top-left header region by default

# Confidence floor for a match to count.
MIN_MATCHES = 8

ref_full = Image.open(REFERENCE_IMAGE).convert("RGB")
tgt_full = Image.open(TARGET_IMAGE).convert("RGB")
print(f"reference: {ref_full.size}")
print(f"target:    {tgt_full.size}")

In [ ]:
x, y, w, h = REFERENCE_BBOX
ref_crop = ref_full.crop((x, y, x + w, y + h))

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(ref_full)
axes[0].add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor="red", linewidth=2))
axes[0].set_title("reference frame + bbox")
axes[0].axis("off")
axes[1].imshow(ref_crop)
axes[1].set_title("reference crop")
axes[1].axis("off")
axes[2].imshow(tgt_full)
axes[2].set_title("target frame (search here)")
axes[2].axis("off")
plt.tight_layout()
plt.show()

## 2. Run SuperPoint + LightGlue

SuperPoint extracts keypoints + descriptors from each image.  
LightGlue matches descriptors between the crop and the target.

In [ ]:
extractor = SuperPoint(max_num_keypoints=2048).eval().to(DEVICE)
matcher = LightGlue(features="superpoint").eval().to(DEVICE)


def pil_to_lightglue(img: Image.Image) -> torch.Tensor:
    """Convert a PIL RGB image to the (3, H, W) float tensor LightGlue expects."""
    arr = np.asarray(img).astype(np.float32) / 255.0
    return torch.from_numpy(arr).permute(2, 0, 1)


ref_tensor = pil_to_lightglue(ref_crop).to(DEVICE)
tgt_tensor = pil_to_lightglue(tgt_full).to(DEVICE)

with torch.inference_mode():
    feats_ref = extractor.extract(ref_tensor)
    feats_tgt = extractor.extract(tgt_tensor)
    matches_out = matcher({"image0": feats_ref, "image1": feats_tgt})

feats_ref, feats_tgt, matches_out = (rbd(x) for x in (feats_ref, feats_tgt, matches_out))
kpts_ref = feats_ref["keypoints"].cpu().numpy()
kpts_tgt = feats_tgt["keypoints"].cpu().numpy()
match_pairs = matches_out["matches"].cpu().numpy()
scores = matches_out["scores"].cpu().numpy()

print(f"keypoints in reference crop: {len(kpts_ref)}")
print(f"keypoints in target frame:   {len(kpts_tgt)}")
print(f"raw matches:                 {len(match_pairs)}")

## 3. Locate the matched region

Take the matched target-frame keypoints, compute their bounding box.  
If we have fewer than `MIN_MATCHES` strong matches, declare the element not found and bail.

In [ ]:
matched_ref = kpts_ref[match_pairs[:, 0]]
matched_tgt = kpts_tgt[match_pairs[:, 1]]

found = len(matched_tgt) >= MIN_MATCHES

if found:
    x_min, y_min = matched_tgt.min(axis=0)
    x_max, y_max = matched_tgt.max(axis=0)
    predicted_bbox = (int(x_min), int(y_min), int(x_max - x_min), int(y_max - y_min))
    print(f"MATCH — {len(matched_tgt)} keypoints, mean score {scores.mean():.3f}")
    print(f"predicted bbox (x, y, w, h): {predicted_bbox}")
else:
    predicted_bbox = None
    print(f"NO MATCH — only {len(matched_tgt)} matches (< {MIN_MATCHES})")
    print("fall back to the next cascade step (OCR / UI-VLM).")

## 4. Visualize

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

axes[0].imshow(ref_crop)
axes[0].scatter(matched_ref[:, 0], matched_ref[:, 1], c="lime", s=12)
axes[0].set_title(f"reference crop — {len(matched_ref)} matched keypoints")
axes[0].axis("off")

axes[1].imshow(tgt_full)
axes[1].scatter(matched_tgt[:, 0], matched_tgt[:, 1], c="lime", s=12)
if found:
    px, py, pw, ph = predicted_bbox
    axes[1].add_patch(patches.Rectangle((px, py), pw, ph, fill=False, edgecolor="red", linewidth=2))
    axes[1].set_title(f"target frame — predicted bbox {predicted_bbox}")
else:
    axes[1].set_title("target frame — no confident match")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 5. Match-line view

Side-by-side with lines connecting each matched keypoint pair. Useful for sanity-checking that the matches are spatially coherent (a real match clusters; noise scatters).

In [ ]:
ref_arr = np.asarray(ref_crop)
tgt_arr = np.asarray(tgt_full)
h_ref, w_ref = ref_arr.shape[:2]
h_tgt, w_tgt = tgt_arr.shape[:2]

canvas_h = max(h_ref, h_tgt)
canvas = np.zeros((canvas_h, w_ref + w_tgt, 3), dtype=np.uint8)
canvas[:h_ref, :w_ref] = ref_arr
canvas[:h_tgt, w_ref : w_ref + w_tgt] = tgt_arr

fig, ax = plt.subplots(figsize=(20, 10))
ax.imshow(canvas)
for (xr, yr), (xt, yt) in zip(matched_ref, matched_tgt):
    ax.plot([xr, xt + w_ref], [yr, yt], c="lime", linewidth=0.5, alpha=0.7)
ax.scatter(matched_ref[:, 0], matched_ref[:, 1], c="red", s=8)
ax.scatter(matched_tgt[:, 0] + w_ref, matched_tgt[:, 1], c="red", s=8)
ax.set_title(f"matches: {len(matched_ref)}")
ax.axis("off")
plt.tight_layout()
plt.show()

## Notes

- **Failure modes worth eyeballing**: matches scattered across the target frame (noise, not a real find); strong cluster in the wrong place (the icon got reused elsewhere); very few matches on icons with little internal structure.
- **Tuning**: bump `max_num_keypoints` for richer scenes, raise `MIN_MATCHES` to be more conservative.
- **Next cascade step** when this fails: OCR for any text label in the recorded crop, then UI-specialized VLM with the crop as a reference image.